In [1]:
from pathlib import Path

model_folder = Path(f"/mount/NAS-workspace-portal/eeg2025-Vistec/models/ml")
data_folder = Path(f"/mount/NAS-workspace-portal/eeg2025-Vistec/models/data")

In [2]:
import ipywidgets as widgets
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Detect available models
def get_available_models(model_folder):
    """Detect all PyTorch model files in the folder."""
    models = []
    if model_folder.exists():
        models = sorted([f.name for f in model_folder.glob("*") if f.is_file()])
    return models

# Detect available data files
def get_available_data(data_folder):
    """Detect Y data files with shape (N, 2) for ACC and RT."""
    data_files = []
    if data_folder.exists():
        for f in sorted(data_folder.glob("Y_*.npy")):
            try:
                data = np.load(f)
                if data.ndim == 2 and data.shape[1] == 2:
                    data_files.append(f.name)
            except Exception:
                pass
    return data_files

def get_available_x_data(data_folder):
    """Detect X data files."""
    x_files = []
    if data_folder.exists():
        x_files = sorted([f.name for f in data_folder.glob("X_*.npy") if f.is_file()])
    return x_files

available_models = get_available_models(model_folder)
available_data = get_available_data(data_folder)
available_x = get_available_x_data(data_folder)

print(f"Found {len(available_models)} models: {available_models}")
print(f"Found {len(available_data)} Y data files: {available_data}")
print(f"Found {len(available_x)} X data files: {available_x}")

Found 2 models: ['evoked_model_500_1771166304.2371216.pkl', 'snr_model_500_1771167871.4968765.pkl']
Found 2 Y data files: ['Y_evoked_500_1771166884.1613965.npy', 'Y_snr_500_1771167865.7485447.npy']
Found 2 X data files: ['X_evoked_500_1771166523.7729328.npy', 'X_snr_500_1771167863.7117176.npy']


In [3]:
# Create widget UI
if available_models:
    model_dropdown = widgets.Dropdown(
        options=available_models,
        description='Model:',
        style={'description_width': '100px'}
    )
else:
    model_dropdown = widgets.Label(value="No models found")

if available_data:
    data_dropdown = widgets.Dropdown(
        options=available_data,
        description='Data (Y):',
        style={'description_width': '100px'}
    )
else:
    data_dropdown = widgets.Label(value="No Y data files found")

if available_x:
    x_dropdown = widgets.Dropdown(
        options=available_x,
        description='Features (X):',
        style={'description_width': '100px'}
    )
else:
    x_dropdown = widgets.Label(value="No X data files found")

predict_button = widgets.Button(
    description='Predict & Plot',
    button_style='info',
    tooltip='Generate predictions and plot True vs Predicted'
)

summary_button = widgets.Button(
    description='Model Summary',
    button_style='warning',
    tooltip='Display model info and parameters'
)

residual_button = widgets.Button(
    description='Residual Analysis',
    button_style='warning',
    tooltip='Plot residuals and error analysis'
)

cv_button = widgets.Button(
    description='Cross-Validate',
    button_style='warning',
    tooltip='Perform cross-validation scoring'
)

output_area = widgets.Output()

# Display widget panel
widgets.VBox([
    widgets.HTML("<h3>Model Tester</h3>"),
    model_dropdown,
    data_dropdown,
    x_dropdown,
    widgets.HBox([predict_button, summary_button, residual_button, cv_button]),
    output_area
])

In [4]:
def on_predict_click(button):
    """Generate predictions and plot True vs Predicted."""
    output_area.clear_output(wait=True)
    
    with output_area:
        try:
            if isinstance(data_dropdown, widgets.Label):
                print("❌ No Y data files available")
                return
            
            selected_data = data_dropdown.value
            data_path = data_folder / selected_data
            
            # Load Y data
            y_true = np.load(data_path)
            print(f"📊 Y data loaded: {selected_data}")
            print(f"   Shape: {y_true.shape} (N={y_true.shape[0]}, features=ACC & RT)")
            print(f"   Accuracy range: [{y_true[:, 0].min():.3f}, {y_true[:, 0].max():.3f}]")
            print(f"   Response Time range: [{y_true[:, 1].min():.3f}, {y_true[:, 1].max():.3f}]")
            
            # Load X data if available
            if not isinstance(x_dropdown, widgets.Label):
                selected_x = x_dropdown.value
                x_path = data_folder / selected_x
                X = np.load(x_path)
                print(f"\n📊 X data loaded: {selected_x}")
                print(f"   Shape: {X.shape} (N={X.shape[0]}, features={X.shape[1]})")
            else:
                X = None
                print(f"\n⚠️  No X data selected")
            
            # Load model if available
            if not isinstance(model_dropdown, widgets.Label) and X is not None:
                selected_model = model_dropdown.value
                model_path = model_folder / selected_model
                
                try:
                    import joblib
                    model = joblib.load(model_path)
                    
                    y_pred = model.predict(X)
                    print(f"\n✅ Model predictions generated")
                    
                    # Create True vs Predicted plots
                    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
                    
                    # Plot 1: True vs Predicted Accuracy
                    axes[0].scatter(y_true[:, 0], y_pred[:, 0], alpha=0.6, s=30)
                    axes[0].plot([y_true[:, 0].min(), y_true[:, 0].max()],
                                [y_true[:, 0].min(), y_true[:, 0].max()],
                                'r--', lw=2, label='Perfect')
                    axes[0].set_xlabel('True Accuracy')
                    axes[0].set_ylabel('Predicted Accuracy')
                    axes[0].set_title('Accuracy: True vs Predicted')
                    axes[0].grid(True, alpha=0.3)
                    axes[0].legend()
                    
                    # Plot 2: True vs Predicted Response Time
                    axes[1].scatter(y_true[:, 1], y_pred[:, 1], alpha=0.6, s=30, color='orange')
                    axes[1].plot([y_true[:, 1].min(), y_true[:, 1].max()],
                                [y_true[:, 1].min(), y_true[:, 1].max()],
                                'r--', lw=2, label='Perfect')
                    axes[1].set_xlabel('True Response Time')
                    axes[1].set_ylabel('Predicted Response Time')
                    axes[1].set_title('Response Time: True vs Predicted')
                    axes[1].grid(True, alpha=0.3)
                    axes[1].legend()
                    
                    plt.tight_layout()
                    plt.show()
                    
                    # Compute metrics
                    corr_acc = np.corrcoef(y_true[:, 0], y_pred[:, 0])[0, 1]
                    corr_rt = np.corrcoef(y_true[:, 1], y_pred[:, 1])[0, 1]
                    mae_acc = np.mean(np.abs(y_true[:, 0] - y_pred[:, 0]))
                    mae_rt = np.mean(np.abs(y_true[:, 1] - y_pred[:, 1]))
                    
                    print(f"\n📈 Metrics:")
                    print(f"  Accuracy  - Corr: {corr_acc:.4f}, MAE: {mae_acc:.4f}")
                    print(f"  RT        - Corr: {corr_rt:.4f}, MAE: {mae_rt:.4f}")
                except Exception as e:
                    print(f"⚠️  Could not load model: {e}")
                    import traceback
                    traceback.print_exc()
            elif X is not None:
                print(f"\nℹ️  No model selected. Showing data distribution only.")
                
                fig, axes = plt.subplots(1, 2, figsize=(14, 5))
                axes[0].hist(y_true[:, 0], bins=20, alpha=0.7, edgecolor='black')
                axes[0].set_xlabel('Accuracy')
                axes[0].set_ylabel('Count')
                axes[0].set_title('Accuracy Distribution')
                axes[0].grid(True, alpha=0.3)
                
                axes[1].hist(y_true[:, 1], bins=20, alpha=0.7, color='orange', edgecolor='black')
                axes[1].set_xlabel('Response Time')
                axes[1].set_ylabel('Count')
                axes[1].set_title('Response Time Distribution')
                axes[1].grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
            
            print(f"\n✨ Done!")
            
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

def on_summary_click(button):
    """Display model info and parameters."""
    output_area.clear_output(wait=True)
    
    with output_area:
        try:
            if isinstance(model_dropdown, widgets.Label):
                print("❌ No model selected")
                return
            
            selected_model = model_dropdown.value
            model_path = model_folder / selected_model
            
            import joblib
            model = joblib.load(model_path)
            
            print(f"📋 Model Summary: {selected_model}")
            print(f"\n🔧 Model Type: {type(model).__name__}")
            print(f"\n📦 Pipeline Steps:")
            if hasattr(model, 'named_steps'):
                for name, step in model.named_steps.items():
                    print(f"  • {name}: {type(step).__name__}")
                    if hasattr(step, 'get_params'):
                        params = step.get_params()
                        for key, val in list(params.items())[:3]:
                            print(f"      - {key}: {val}")
                        if len(params) > 3:
                            print(f"      ... and {len(params) - 3} more")
            else:
                print(f"  {type(model).__name__}")
                if hasattr(model, 'get_params'):
                    params = model.get_params()
                    print(f"\n⚙️ Parameters:")
                    for key, val in list(params.items())[:5]:
                        print(f"  • {key}: {val}")
                    if len(params) > 5:
                        print(f"  ... and {len(params) - 5} more")
            
            # Estimate model size
            import pickle
            model_bytes = len(pickle.dumps(model))
            print(f"\n💾 Model Size: {model_bytes / (1024*1024):.2f} MB")
            
            print(f"\n✨ Done!")
            
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

def on_residual_click(button):
    """Plot residuals and error analysis."""
    output_area.clear_output(wait=True)
    
    with output_area:
        try:
            if isinstance(data_dropdown, widgets.Label) or isinstance(model_dropdown, widgets.Label):
                print("❌ Model or Y data not selected")
                return
            
            selected_data = data_dropdown.value
            selected_x = x_dropdown.value if not isinstance(x_dropdown, widgets.Label) else None
            
            if selected_x is None:
                print("❌ X data not selected")
                return
            
            data_path = data_folder / selected_data
            x_path = data_folder / selected_x
            
            y_true = np.load(data_path)
            X = np.load(x_path)
            
            import joblib
            model_path = model_folder / model_dropdown.value
            model = joblib.load(model_path)
            
            y_pred = model.predict(X)
            
            # Calculate residuals
            residuals_acc = y_true[:, 0] - y_pred[:, 0]
            residuals_rt = y_true[:, 1] - y_pred[:, 1]
            
            print(f"📊 Residual Analysis")
            print(f"\nAccuracy Residuals:")
            print(f"  Mean: {np.mean(residuals_acc):.4f}")
            print(f"  Std:  {np.std(residuals_acc):.4f}")
            print(f"  Min:  {np.min(residuals_acc):.4f}")
            print(f"  Max:  {np.max(residuals_acc):.4f}")
            
            print(f"\nResponse Time Residuals:")
            print(f"  Mean: {np.mean(residuals_rt):.4f}")
            print(f"  Std:  {np.std(residuals_rt):.4f}")
            print(f"  Min:  {np.min(residuals_rt):.4f}")
            print(f"  Max:  {np.max(residuals_rt):.4f}")
            
            # Plot residuals
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            
            # Residual scatter ACC
            axes[0, 0].scatter(y_pred[:, 0], residuals_acc, alpha=0.6, s=30)
            axes[0, 0].axhline(y=0, color='r', linestyle='--', lw=2)
            axes[0, 0].set_xlabel('Predicted Accuracy')
            axes[0, 0].set_ylabel('Residuals')
            axes[0, 0].set_title('Accuracy Residuals vs Predicted')
            axes[0, 0].grid(True, alpha=0.3)
            
            # Residual scatter RT
            axes[0, 1].scatter(y_pred[:, 1], residuals_rt, alpha=0.6, s=30, color='orange')
            axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
            axes[0, 1].set_xlabel('Predicted Response Time')
            axes[0, 1].set_ylabel('Residuals')
            axes[0, 1].set_title('RT Residuals vs Predicted')
            axes[0, 1].grid(True, alpha=0.3)
            
            # Residual histogram ACC
            axes[1, 0].hist(residuals_acc, bins=20, alpha=0.7, edgecolor='black')
            axes[1, 0].set_xlabel('Residuals')
            axes[1, 0].set_ylabel('Count')
            axes[1, 0].set_title('Accuracy Residuals Distribution')
            axes[1, 0].grid(True, alpha=0.3)
            
            # Residual histogram RT
            axes[1, 1].hist(residuals_rt, bins=20, alpha=0.7, edgecolor='black', color='orange')
            axes[1, 1].set_xlabel('Residuals')
            axes[1, 1].set_ylabel('Count')
            axes[1, 1].set_title('RT Residuals Distribution')
            axes[1, 1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            print(f"\n✨ Done!")
            
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

def on_cv_click(button):
    """Perform cross-validation scoring."""
    output_area.clear_output(wait=True)
    
    with output_area:
        try:
            if isinstance(model_dropdown, widgets.Label) or isinstance(data_dropdown, widgets.Label):
                print("❌ Model or data not selected")
                return
            
            selected_x = x_dropdown.value if not isinstance(x_dropdown, widgets.Label) else None
            if selected_x is None:
                print("❌ X data not selected")
                return
            
            selected_data = data_dropdown.value
            selected_x = x_dropdown.value
            
            data_path = data_folder / selected_data
            x_path = data_folder / selected_x
            
            y = np.load(data_path)
            X = np.load(x_path)
            
            import joblib
            from sklearn.model_selection import KFold
            
            model_path = model_folder / model_dropdown.value
            model = joblib.load(model_path)
            
            print(f"🔄 Cross-Validation (5-Fold)")
            
            kf = KFold(n_splits=5, shuffle=True, random_state=42)
            
            cv_corr_acc = []
            cv_corr_rt = []
            cv_mae_acc = []
            cv_mae_rt = []
            
            for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
                X_test, y_test = X[test_idx], y[test_idx]
                y_pred = model.predict(X_test)
                
                corr_acc = np.corrcoef(y_test[:, 0], y_pred[:, 0])[0, 1]
                corr_rt = np.corrcoef(y_test[:, 1], y_pred[:, 1])[0, 1]
                mae_acc = np.mean(np.abs(y_test[:, 0] - y_pred[:, 0]))
                mae_rt = np.mean(np.abs(y_test[:, 1] - y_pred[:, 1]))
                
                cv_corr_acc.append(corr_acc)
                cv_corr_rt.append(corr_rt)
                cv_mae_acc.append(mae_acc)
                cv_mae_rt.append(mae_rt)
                
                print(f"  Fold {fold}: ACC (r={corr_acc:.4f}, mae={mae_acc:.4f}) | RT (r={corr_rt:.4f}, mae={mae_rt:.4f})")
            
            print(f"\n📊 Cross-Validation Summary:")
            print(f"\nAccuracy:")
            print(f"  Mean Corr: {np.mean(cv_corr_acc):.4f} ± {np.std(cv_corr_acc):.4f}")
            print(f"  Mean MAE:  {np.mean(cv_mae_acc):.4f} ± {np.std(cv_mae_acc):.4f}")
            
            print(f"\nResponse Time:")
            print(f"  Mean Corr: {np.mean(cv_corr_rt):.4f} ± {np.std(cv_corr_rt):.4f}")
            print(f"  Mean MAE:  {np.mean(cv_mae_rt):.4f} ± {np.std(cv_mae_rt):.4f}")
            
            # Plot CV scores
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            folds = np.arange(1, 6)
            
            axes[0].plot(folds, cv_corr_acc, 'o-', label='Accuracy', linewidth=2, markersize=8)
            axes[0].plot(folds, cv_corr_rt, 's-', label='Response Time', linewidth=2, markersize=8, color='orange')
            axes[0].set_xlabel('Fold')
            axes[0].set_ylabel('Correlation')
            axes[0].set_title('Cross-Validation Correlations')
            axes[0].grid(True, alpha=0.3)
            axes[0].legend()
            axes[0].set_xticks(folds)
            
            axes[1].plot(folds, cv_mae_acc, 'o-', label='Accuracy', linewidth=2, markersize=8)
            axes[1].plot(folds, cv_mae_rt, 's-', label='Response Time', linewidth=2, markersize=8, color='orange')
            axes[1].set_xlabel('Fold')
            axes[1].set_ylabel('MAE')
            axes[1].set_title('Cross-Validation MAE')
            axes[1].grid(True, alpha=0.3)
            axes[1].legend()
            axes[1].set_xticks(folds)
            
            plt.tight_layout()
            plt.show()
            
            print(f"\n✨ Done!")
            
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

# Register button handlers
predict_button.on_click(on_predict_click)
summary_button.on_click(on_summary_click)
residual_button.on_click(on_residual_click)
cv_button.on_click(on_cv_click)